# Home Repair Trend Pipeline — v0.1
**Sources:** YouTube Data API v3 · Google Trends (pytrends)  
**Goal:** Score a list of seed topics and surface the fastest-growing ones.

---
### Setup checklist
- [ ] YouTube Data API v3 enabled in Google Cloud Console
- [ ] API key created at https://console.cloud.google.com/apis/credentials
- [ ] Set `YOUTUBE_API_KEY` below (or in a `.env` file)
- [ ] pytrends requires no key — it uses a browser-like session

In [ ]:
# Cell 1 — Install dependencies
# Run once, then you can comment this out
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'google-api-python-client',
    'pytrends',
    'pandas',
    'python-dotenv'
])
print('Dependencies installed.')

In [ ]:
# Cell 2 — Configuration
# Option A: paste key directly (fine for local dev, never commit to git)
# Option B: use a .env file (recommended for GitHub)

import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env if present

# Set your key here OR in a .env file as: YOUTUBE_API_KEY=AIza...
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY', 'PASTE_YOUR_KEY_HERE')

if YOUTUBE_API_KEY == 'PASTE_YOUR_KEY_HERE':
    print('⚠️  No YouTube API key found. Set YOUTUBE_API_KEY in .env or above.')
else:
    print(f'✅ YouTube API key loaded: {YOUTUBE_API_KEY[:8]}...')

# Scoring weights — tune these as you learn which signals you trust most
WEIGHT_YOUTUBE  = 0.55
WEIGHT_TRENDS   = 0.45

# How many YouTube videos to sample per topic
YT_RESULTS_PER_TOPIC = 10

# Trends: compare this recent window vs. a longer baseline
TRENDS_RECENT    = 'today 1-m'   # last 30 days
TRENDS_BASELINE  = 'today 3-m'   # last 90 days

In [ ]:
# Cell 3 — Seed topics
# These are the topics you want to score. Expand this list over time.
# Keep them specific — vague topics like 'home repair' return noisy results.

SEED_TOPICS = [
    'garage door not closing',
    'water heater making noise',
    'bathroom exhaust fan replacement',
    'drywall hole repair',
    'leaking shower grout fix',
    'toilet running constantly',
    'exterior door weatherstripping',
    'HVAC filter replacement',
    'circuit breaker tripping',
    'roof shingle damage assessment',
]

print(f'Scoring {len(SEED_TOPICS)} topics.')

In [ ]:
# Cell 4 — YouTube: measure video velocity
#
# Strategy: search each topic, get the top N videos, compare recent
# view counts against older ones to compute a velocity score.
# Recent videos (< 90 days) with high views = fast-growing topic.

from googleapiclient.discovery import build
from datetime import datetime, timedelta, timezone
import pandas as pd
import time

def build_youtube_client(api_key: str):
    return build('youtube', 'v3', developerKey=api_key)

def fetch_youtube_videos(client, query: str, max_results: int = 10) -> list[dict]:
    """
    Search YouTube for a query, return a list of video metadata dicts.
    Cost: 100 units per search call + 1 unit per video in the stats call.
    With 10k daily free quota and 10 topics x 100 units = 1,000 units — very safe.
    """
    # Step 1: search for video IDs
    search_resp = client.search().list(
        q=query,
        part='id,snippet',
        type='video',
        maxResults=max_results,
        order='viewCount',          # surface most-watched first
        relevanceLanguage='en',
    ).execute()

    video_ids = [
        item['id']['videoId']
        for item in search_resp.get('items', [])
        if item['id'].get('videoId')
    ]

    if not video_ids:
        return []

    # Step 2: get statistics for those video IDs
    stats_resp = client.videos().list(
        id=','.join(video_ids),
        part='statistics,snippet',
    ).execute()

    videos = []
    for item in stats_resp.get('items', []):
        published_raw = item['snippet'].get('publishedAt', '')
        try:
            published_at = datetime.fromisoformat(published_raw.replace('Z', '+00:00'))
        except ValueError:
            published_at = None

        view_count = int(item['statistics'].get('viewCount', 0))

        videos.append({
            'video_id':    item['id'],
            'title':       item['snippet']['title'],
            'published_at': published_at,
            'view_count':  view_count,
            'channel':     item['snippet'].get('channelTitle', ''),
        })

    return videos


def compute_youtube_velocity(videos: list[dict], recent_days: int = 90) -> float:
    """
    Velocity = (avg views/day for recent videos) / (avg views/day for older videos + 1)
    Capped and normalized to 0-1.

    A topic where recent videos are outperforming older ones scores high.
    We add 1 to the denominator to avoid divide-by-zero.
    """
    if not videos:
        return 0.0

    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(days=recent_days)

    recent, older = [], []
    for v in videos:
        if v['published_at'] is None:
            continue
        age_days = max((now - v['published_at']).days, 1)
        vpd = v['view_count'] / age_days  # views per day
        if v['published_at'] >= cutoff:
            recent.append(vpd)
        else:
            older.append(vpd)

    avg_recent = sum(recent) / len(recent) if recent else 0
    avg_older  = sum(older)  / len(older)  if older  else 0

    # Raw ratio — topics where recent >> older score high
    ratio = avg_recent / (avg_older + 1)

    # Normalize: sigmoid-like cap so outliers don't dominate
    # A ratio of 2x maps to ~0.67, 5x maps to ~0.83, 10x to ~0.91
    normalized = ratio / (ratio + 1)

    return round(min(normalized, 1.0), 4)


print('YouTube functions defined.')

In [ ]:
# Cell 5 — Google Trends: measure growth
#
# pytrends has no API key — it scrapes Google Trends like a browser.
# It is rate-limited by Google, so we add a sleep between requests.
# The pytrends repo was archived in April 2025 but still works for
# light use. We'll build the pipeline so it's easy to swap in an
# official source later.

from pytrends.request import TrendReq
import numpy as np

def build_trends_client() -> TrendReq:
    return TrendReq(
        hl='en-US',
        tz=360,
        timeout=(10, 30),
        retries=2,
        backoff_factor=0.5,
    )


def compute_trends_growth(pytrends_client: TrendReq, keyword: str) -> float:
    """
    Fetch 90 days of interest-over-time data and compute:
        growth = avg(last 30 days) / avg(prior 60 days)
    Normalized to 0-1.

    Returns 0.0 on any error (rate limit, no data, etc.).
    """
    try:
        pytrends_client.build_payload(
            kw_list=[keyword],
            timeframe='today 3-m',   # 90 days
            geo='US',
        )
        df = pytrends_client.interest_over_time()

        if df.empty or keyword not in df.columns:
            return 0.0

        values = df[keyword].values.astype(float)
        if len(values) < 8:
            return 0.0

        # Split into recent (last third) vs baseline (first two thirds)
        split = len(values) * 2 // 3
        baseline_avg = np.mean(values[:split]) + 0.01   # +0.01 avoids div/0
        recent_avg   = np.mean(values[split:])

        ratio = recent_avg / baseline_avg

        # Same normalization as YouTube velocity
        normalized = ratio / (ratio + 1)
        return round(min(normalized, 1.0), 4)

    except Exception as e:
        print(f'  Trends error for "{keyword}": {e}')
        return 0.0


print('Trends functions defined.')

In [ ]:
# Cell 6 — Run the pipeline
#
# This loops through every seed topic, collects both signals,
# and builds the scored results list.
#
# Expect ~3-5 seconds per topic due to API calls and rate-limit sleeps.
# For 10 topics: roughly 30-60 seconds total.

results = []

yt_client     = build_youtube_client(YOUTUBE_API_KEY)
trends_client = build_trends_client()

for i, topic in enumerate(SEED_TOPICS):
    print(f'[{i+1}/{len(SEED_TOPICS)}] Scoring: "{topic}"')

    # --- YouTube ---
    yt_videos  = fetch_youtube_videos(yt_client, topic, max_results=YT_RESULTS_PER_TOPIC)
    yt_velocity = compute_youtube_velocity(yt_videos)
    print(f'       YT velocity:    {yt_velocity:.3f}  ({len(yt_videos)} videos fetched)')

    # --- Google Trends ---
    # Sleep to avoid Google rate-limiting us between Trends requests
    time.sleep(3)
    trends_growth = compute_trends_growth(trends_client, topic)
    print(f'       Trends growth:  {trends_growth:.3f}')

    # --- Combined score ---
    score = round(
        yt_velocity  * WEIGHT_YOUTUBE +
        trends_growth * WEIGHT_TRENDS,
        4
    )
    print(f'       Combined score: {score:.3f}')
    print()

    results.append({
        'topic':            topic,
        'youtube_velocity': yt_velocity,
        'trends_growth':    trends_growth,
        'score':            score,
        'yt_video_count':   len(yt_videos),
    })

print('Pipeline complete.')

In [ ]:
# Cell 7 — Results table

df_results = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)

print('=== TREND SCORES (sorted by score) ===')
print()
print(df_results[['topic', 'youtube_velocity', 'trends_growth', 'score']].to_string(index=True))

In [ ]:
# Cell 8 — JSON output (matches the format from our pipeline spec)

import json

output = df_results[['topic', 'youtube_velocity', 'trends_growth', 'score']].to_dict(orient='records')

print(json.dumps(output, indent=2))

# Optionally save to file
with open('trend_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('\nSaved to trend_results.json')

In [ ]:
# Cell 9 — Quick inspection: what videos did we find for the top topic?
# Useful for sanity-checking that the YouTube results are relevant.

top_topic = df_results.iloc[0]['topic']
print(f'Top scoring topic: "{top_topic}"\n')
print('Fetching video sample for inspection...')

sample_videos = fetch_youtube_videos(yt_client, top_topic, max_results=5)

for v in sample_videos:
    age = (datetime.now(timezone.utc) - v['published_at']).days if v['published_at'] else '?'
    print(f"  [{age}d ago]  {v['view_count']:>10,} views  —  {v['title'][:70]}")

---
## Notes for v0.2

**Things to improve next:**
- Add a volume floor filter: topics with fewer than N total views should be excluded before scoring (prevents low-volume topics from gaming the velocity metric)
- Store runs in a SQLite or JSON log so you can track scores week-over-week
- Add Reddit signal once API access is approved
- Swap pytrends for the official Google Trends API (currently in alpha) once it opens up
- Add a `.gitignore` entry for `.env` before pushing to GitHub

**GitHub checklist before pushing:**
- [ ] Add `.env` to `.gitignore` (never commit API keys)
- [ ] Include a `requirements.txt` with pinned versions
- [ ] Include a `README.md` describing the pipeline and how to set up credentials
- [ ] The `trend_results.json` output file is safe to commit